<a href="https://colab.research.google.com/github/DANFELcode/IAIA-L06/blob/main/L06_AZUL_IAIA_2026_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INSTRUCCIONES

## OBJETIVOS

Desarrollar competencias básicas para:

1. Comprender qué es una **ontología** y para qué sirve representar conocimiento con clases, propiedades e individuos.
2. Modelar una ontología separando lo **abstracto** (clases y propiedades de clases) de lo **concreto** (individuos).
3. Distinguir las dos formas de propiedad: **dato simple** (tipo número, texto, booleano) y **relación con individuos de otra clase**.
4. Razonar automáticamente sobre una ontología para **clasificar** individuos a partir de las definiciones modeladas.
5. Definir **reglas de inferencia** sobre una ontología y reconocer su parecido con cláusulas tipo Horn (estilo Prolog).

## ENTREGABLE

*Reglas para el envío de los entregables:*

- **Forma de envío:**

  Este laboratorio se debe enviar únicamente a través de la plataforma [Moodle](https://campusvirtual.escuelaing.edu.co/moodle/) en la actividad definida.

- **Archivo de entrega:**

  Incluyan en un archivo `.zip` los archivos correspondientes al laboratorio, revise **[How to cite in Jupyter Notebooks](https://gist.github.com/ELS4NTA/2c188b3b7300a726e60d71a58a55f740)** para citar cualquier recurso utilizado y ver la estructura del proyecto.

- **Nomenclatura para nombrar el archivo de entrega:**

  El archivo deberá ser renombrado en mayúsculas con el siguiente formato: `L06-{color_grupo}-IAIA-{semestre_academico}.zip`

  Ejemplos:
  ```
  L06-AZUL-IAIA-2026-1.zip
  L06-ROJO-IAIA-2026-1.zip
  ```

Revise cuando lo requiera <a href="https://aima.cs.berkeley.edu/">(Russell & Norvig, 2021, cap. 10) <cite data-cite="russell2021artificial"></cite></a>.

## Herramientas del laboratorio

| Librería | Para qué la usaremos | Documentación |
|---|---|---|
| [`owlready2`](https://owlready2.readthedocs.io/) | Construir ontologías OWL 2.0 desde Python, razonar con `sync_reasoner_pellet` y definir reglas `Imp` (SWRL) | <https://owlready2.readthedocs.io/> |

> **Requisito previo — Java**. El razonador interno de `owlready2`
> (`sync_reasoner_pellet`) y las reglas `Imp` corren sobre **Pellet**, que
> es un programa Java. Verifiquen tener Java instalado:
>
> ```bash
> java -version
> ```
>
> Si no lo tienen, instálenlo desde [java.com](https://www.java.com/) (Win/Mac) o con el gestor de paquetes de Linux (paquetes `jre` o `jdk`).

## Antes de empezar

Cuando trabajen con ontologías van a ver constantemente tres nombres: **OWL**, **RDF** y **Turtle**. Esta tabla los aclara antes de empezar.

| Sigla / formato | ¿Qué es? | ¿Para qué sirve? |
|---|---|---|
| **RDF** (Resource Description Framework) | Modelo abstracto de datos. Toda afirmación se escribe como una **tripleta** `(sujeto, predicado, objeto)`. No es un archivo, es la *forma* en la que se piensa el conocimiento. | Es el modelo de fondo: cualquier ontología termina, conceptualmente, como un conjunto de tripletas RDF. |
| **OWL** (Web Ontology Language) | Lenguaje formal *encima* de RDF que añade vocabulario para **clases**, **propiedades**, **restricciones** y **reglas**. | Es el lenguaje en el que escribimos la ontología. Todo lo que harán en este lab — `class Leon(Mamifero): pass`, `simba.come.append(dolly)`, reglas `Imp` — es OWL. |
| **Turtle** (`.ttl`) | Formato de texto compacto y legible para escribir tripletas RDF. | El que usaremos para *guardar* la ontología y poder inspeccionarla en un editor. |

**En síntesis**: RDF es el *modelo de datos* (tripletas), OWL es el *vocabulario* (clases, propiedades, axiomas, reglas) y Turtle es solo una *forma de escribirlo en un archivo*.

# I. DESARROLLANDO ONTOLOGÍAS

En esta sección construirán, paso a paso, una **ontología mínima** sobre
animales para conocer los constructos básicos de `owlready2`.

Al terminar la sección habrán modelado las dos partes de una ontología:
las **clases y propiedades** (lo abstracto) y los **individuos** (lo
concreto).

## A. EXPLICACIÓN

Una **ontología** es una representación formal de un dominio: nombra las **categorías** (clases) que existen, las **propiedades** que las relacionan y los **individuos** concretos que pertenecen a cada categoría. Es un vocabulario con reglas explícitas que un programa puede procesar — no solo un texto para humanos.

### Las partes de una ontología

Una ontología se descompone en dos planos: **lo abstracto** (las categorías
y las propiedades que pueden tenerse) y **lo concreto** (los individuos
particulares que existen en ese mundo).

| Plano | Elementos OWL | Rol gramatical | Ejemplo del lab |
|---|---|---|---|
| **Abstracto** | **Clases** | sustantivos / categorías | `Animal`, `Mamifero`, `Leon`, `Oveja` |
| **Abstracto** | **Propiedades** | verbos | `tiene_pelos`, `come` |
| **Concreto** | **Individuos** | nombres propios | `simba` (un león), `dolly` (una oveja) |

### Propiedad

Dentro del plano abstracto, una propiedad puede tomar dos formas según
con qué conecte al individuo:

| Tipo en `owlready2` | Conecta al individuo con… | Ejemplo |
|---|---|---|
| **`DataProperty`** | un **dato simple** (booleano, número, texto, fecha) | `simba.tiene_pelos = True` — un león *tiene pelos*. |
| **`ObjectProperty`** | otro **individuo** (de la misma o de otra clase) | `simba.come.append(dolly)` — el león *come* a la oveja. |

Esta distinción es uno de los conceptos centrales del laboratorio: cuando
modelen su propia ontología tendrán que decidir, para
cada atributo del dominio, si es un `DataProperty` o un `ObjectProperty`.

### Identidad por IRI

Cada clase, individuo y propiedad vive en un **IRI** (Internationalized
Resource Identifier) único, formado por el *namespace* de la ontología
(`http://escuelaing.edu.co/iaia/lab06/animales.owl#`) más el nombre local.
Por eso `get_ontology(IRI)` es lo primero que hacemos: define el espacio
de nombres del que nacerán todos los identificadores.

### `Thing` y las jerarquías de subcategorías

En OWL todas las clases son, en última instancia, **subclases** de una
clase predefinida llamada `Thing`. Cuando ustedes escriben
`class Animal(Thing): pass`, están afirmando que `Animal` es una categoría
de la ontología. Cuando luego escriben `class Mamifero(Animal): pass`, están
afirmando que `Mamifero` es una **subclase** de `Animal` — todo mamífero
es un animal.

### Tres operaciones centrales sobre categorías

| Operación | ¿Qué afirma? | Ejemplo en este lab | ¿Quién la introduce? |
|---|---|---|---|
| **Subsumción** | Una clase es subclase de otra: `C ⊑ D`. | `Mamifero ⊑ Animal`, `Leon ⊑ Mamifero` | Ustedes, al escribir `class Leon(Mamifero): pass`. |
| **Instanciación** | Un individuo pertenece a una clase: `x : C`. | `simba : Leon` | Ustedes, al escribir `simba = Leon("simba")`. |
| **Clasificación** | El razonador *deriva* que un individuo pertenece a una clase, a partir de sus propiedades y de una regla. | Si una regla SWRL dice `Animal(?x), come(?x, ?y), Animal(?y) -> Carnivoro(?x)`, entonces `simba` es clasificado automáticamente como `Carnivoro` cuando se afirma `simba.come.append(dolly)`. | El razonador `sync_reasoner_pellet` de `owlready2`. |

Estas tres operaciones son el hilo conductor del laboratorio: la
**subsumción** y la **instanciación** las afirman ustedes al modelar; la
**clasificación** la hará el razonador más adelante.

## B. REPRESENTACIÓN

Modelen, paso a paso, la frase guía:

> *"Un león es un mamífero. Un león tiene pelos. Un león come ovejas. Simba es un león."*

Empezarán por lo **abstracto** — clases (`Animal`, `Mamifero`, `Leon`,
`Oveja`) y propiedades (`tiene_pelos`, `come`) — y terminarán por lo
**concreto** — el individuo `simba` con sus datos.

> Para el detalle de cada construcción que aparece en los pasos siguientes
> (clases, propiedades, individuos), Lamy
> <cite data-cite="lamy2021ontologies"></cite> cap. 5, secciones 5.1–5.6
> sirve como manual de referencia.

### 0. Imports y ontología vacía

Importen `owlready2` y creen una ontología vacía. El **IRI** que pasen a
`get_ontology` será el namespace donde nacerán todas las clases,
propiedades e individuos que escriban a continuación.

```python
with onto:
    <Python code>
```

Todo lo que escriban dentro del bloque `with onto: ...` se añadirá a la
ontología `onto`.

In [1]:
!pip install owlready2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.4/27.4 MB 30.9 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for owlready2: filename=owlready2-0.51-cp313-cp313-linux_x86_64.whl size=24780856 sha256=4245d454babc1ffb462b89fe3e5f21ced25141a068bd2e1b77d3543719e52b37
  Stored in directory: /root/.cache/pip/wheels/cc/16/52/762fdfa8c53f256db44674790a3ceba702c3c763c96ce8b62c
Successfully built owlready2


In [2]:
# **COMPLETE** — importar owlready2 y crear la ontología vacía.
from owlready2 import *

ONTO_IRI = "http://escuelaing.edu.co/iaia/lab06/animales.owl"
onto = get_ontology(ONTO_IRI)
onto

get_ontology("http://escuelaing.edu.co/iaia/lab06/animales.owl#")

### 1. La primera clase abstracta

Definan la clase `Animal` como subclase de `Thing`. Toda clase, propiedad o
individuo de la ontología debe crearse dentro del bloque `with onto: ...`.

> *"Un animal es una cosa."* — afirmación implícita de toda ontología:
> todas las clases viven debajo de `Thing`.

In [3]:
with onto:
    class Animal(Thing):
        pass

### 2. Subclases: león y oveja son mamíferos

Modelen ahora las dos frases *"un león es un mamífero"* y *"una oveja
es un mamífero"*. Necesitan tres subclases anidadas:

- `Mamifero ⊑ Animal`
- `Leon ⊑ Mamifero`
- `Oveja ⊑ Mamifero`

Declaren además que `Leon` y `Oveja` son **disjuntas**: ningún individuo
puede ser a la vez león y oveja.

In [4]:
with onto:
    class Mamifero(Animal):
        pass

    class Leon(Mamifero):
        pass

    class Oveja(Mamifero):
        pass

    AllDisjoint([Leon, Oveja])

### 3. Propiedades — los "verbos" de la ontología

Toca modelar las dos afirmaciones que faltan: *"un león tiene pelos"* y
*"un león come ovejas"*. Cada una corresponde a un sabor distinto de
propiedad:

| Afirmación | Tipo de propiedad | Por qué |
|---|---|---|
| *"un león tiene pelos"* | **`DataProperty`** | conecta al individuo con un **dato simple** (booleano `True`/`False`). |
| *"un león come ovejas"* | **`ObjectProperty`** | conecta al individuo con **otro individuo** de otra clase. |

Modelen las dos propiedades:

| Propiedad | Tipo | Características |
|---|---|---|
| `tiene_pelos` | `DataProperty`, `FunctionalProperty` | dominio `Animal`, rango `bool`. *Funcional* significa "un único valor por individuo". |
| `come` | `ObjectProperty` | dominio `Animal`, rango `Animal`. Codifica la afirmación general de que un animal se alimenta de otro animal — más adelante una regla la usará para clasificar carnívoros. |

In [5]:
with onto:
    class tiene_pelos(DataProperty, FunctionalProperty):
        domain = [Animal]
        range  = [bool]

    class come(ObjectProperty):
        domain = [Animal]
        range  = [Animal]

### 4. Individuos — del plano abstracto al concreto

Hasta aquí toda la ontología es **abstracta**: dice qué clases existen y
qué propiedades pueden tenerse, pero todavía no afirma que exista *ningún*
león o *ninguna* oveja en particular. Toca aterrizar la última afirmación
de la frase guía:

> *"Simba es un león."*  → un **individuo** concreto de la clase `Leon`.

Modelen:

- `simba`, instancia de `Leon`.
- `dolly`, instancia de `Oveja`.
- Asignen los datos del león: `simba.tiene_pelos = True` y `simba.come.append(dolly)`.

In [6]:
with onto:
    simba = Leon("simba")
    dolly = Oveja("dolly")

    simba.tiene_pelos = True
    simba.come.append(dolly)

### 5. Guardar la ontología en Turtle

`owlready2` puede serializar la ontología a un archivo. Guárdenla en
formato **Turtle** (`.ttl`) — el formato legible para humanos — y abran el
archivo resultante en el editor: verán las afirmaciones que ustedes
escribieron, expresadas como tripletas RDF.

In [7]:
onto.save(file="animales.ttl", format="ntriples") # owlready2 escribe en ntriples/turtle
print("¡Ontología guardada exitosamente como animales.ttl!")

¡Ontología guardada exitosamente como animales.ttl!


# II. APLICANDO: ONTOLOGÍA DE PIZZAS

En la sección anterior modelaron una ontología pequeña sin invocar al razonador. Aquí construirán **desde cero** una ontología de pizzas y, por primera vez, clasificarán individuos automáticamente usando **reglas `Imp`** y el razonador `sync_reasoner_pellet`.

El objetivo es que apliquen las tres operaciones —**subsunción, instanciación y clasificación**— en un dominio nuevo, distinguiendo lo abstracto (clases, propiedades, reglas) de lo concreto (pizzas particulares con sus ingredientes y su precio).

## Modelado abstracto: clases y propiedades

Modelen el siguiente vocabulario en una **ontología nueva** (no la mezclen con `animales.ttl`):

### Clases

| Clase | Detalle |
|---|---|
| `Pizza` | clase base |
| `Ingrediente` | clase base |
| `Queso`, `Carne`, `Verdura`, `Salsa` | subclases **disjuntas** de `Ingrediente` |
| `IngredientePicante` | subclase de `Ingrediente`. **No** es disjunta con las anteriores: un jalapeño es a la vez `Verdura` e `IngredientePicante`. |
| `ContieneCarne`, `Picante`, `PizzaMixta`, `PizzaPremium` | subclases de `Pizza`. Quedan **vacías** al modelar — las llenarán las reglas más adelante. |

### Propiedades

| Propiedad | Tipo | Detalle |
|---|---|---|
| `tiene_ingrediente` | `ObjectProperty` | dominio `Pizza`, rango `Ingrediente` |
| `tiene_precio` | `DataProperty`, `FunctionalProperty` | dominio `Pizza`, rango `float` |

In [8]:
from owlready2 import *

# Profe no se si tocaba crear un nuevo esapcio, entonces lo cree ahi
piz = get_ontology("http://escuelaing.edu.co/iaia/lab06/pizzas.owl")

with piz:
    class Pizza(Thing):
        pass

    class Ingrediente(Thing):
        pass

    class Queso(Ingrediente):
        pass

    class Carne(Ingrediente):
        pass

    class Verdura(Ingrediente):
        pass

    class Salsa(Ingrediente):
        pass

    AllDisjoint([Queso, Carne, Verdura, Salsa])

    class IngredientePicante(Ingrediente):
        pass

    class ContieneCarne(Pizza):
        pass

    class Picante(Pizza):
        pass

    class PizzaMixta(Pizza):
        pass

    class PizzaPremium(Pizza):
        pass

    class tiene_ingrediente(ObjectProperty):
        domain = [Pizza]
        range  = [Ingrediente]

    class tiene_precio(DataProperty, FunctionalProperty):
        domain = [Pizza]
        range  = [float]

## Reglas `Imp` (SWRL): el lenguaje de las reglas

La clasificación de pizzas se hará con **reglas tipo `Imp`** — el constructor de `owlready2` para definir reglas en **SWRL** (Semantic Web Rule Language). Una regla SWRL tiene la forma estándar **antecedente → consecuente**:

```
Mamifero(?x), tiene_pelos(?x, true) -> Peludo(?x)
```

Las variables empiezan con `?`, los átomos se separan con `,` (conjunción) y `->` separa antecedente de consecuente. Las variables se entienden **universalmente cuantificadas**.

### Tres notaciones, una semántica

La misma regla se puede escribir en tres lenguajes equivalentes:

| Notación | Forma de la regla |
|---|---|
| **FOL** | $\forall x.\; \text{Mamifero}(x) \land \text{tiene\_pelos}(x, \text{true}) \to \text{Peludo}(x)$ |
| **Prolog** | `peludo(X) :- mamifero(X), tiene_pelos(X, true).` |
| **SWRL en `owlready2`** | `Mamifero(?x), tiene_pelos(?x, true) -> Peludo(?x)` |

En esencia, SWRL es una sintaxis estándar para escribir **cláusulas tipo Horn** de FOL dentro de una ontología OWL <cite data-cite="horrocks2004swrl"></cite>. En `owlready2` cada regla se construye con `Imp().set_as_rule("...")`. Lamy <cite data-cite="lamy2021ontologies"></cite> sec. 7.6 cubre SWRL en detalle, incluyendo los **built-ins** que usaremos en R4.

### Las cuatro reglas del lab

Definan estas cuatro reglas `Imp` sobre la ontología de pizzas:

| # | Regla | Lectura |
|---|---|---|
| R1 | `Pizza(?p), tiene_ingrediente(?p, ?i), Carne(?i) -> ContieneCarne(?p)` | toda pizza con al menos un ingrediente de tipo carne, contiene carne |
| R2 | `Pizza(?p), tiene_ingrediente(?p, ?i), IngredientePicante(?i) -> Picante(?p)` | una pizza con al menos un ingrediente picante es picante |
| R3 | `ContieneCarne(?p), tiene_ingrediente(?p, ?i), Verdura(?i) -> PizzaMixta(?p)` | una pizza ya clasificada como `ContieneCarne` y que además lleva verdura, es mixta |
| R4 | `Pizza(?p), tiene_precio(?p, ?x), greaterThan(?x, 25) -> PizzaPremium(?p)` | una pizza que cuesta más de **25 mil pesos** es premium |

**Encadenamiento.** Fíjense en R3: su antecedente incluye `ContieneCarne(?p)`, que **no** es un hecho que escriben a mano — lo deriva R1. R3 solo puede disparar después de que R1 haya disparado. Esto es *forward chaining*: las reglas se alimentan unas a otras hasta que no haya nada nuevo que derivar.

**Built-in `greaterThan`.** R4 estrena un componente nuevo de SWRL: los **built-ins**. Son predicados predefinidos sobre tipos de datos (números, strings, fechas) que permiten condiciones como "mayor que" o "concatenar strings". Un built-in se escribe como un átomo más en el cuerpo de la regla.

In [9]:
with piz:
    Imp().set_as_rule("Pizza(?p), tiene_ingrediente(?p, ?i), Carne(?i) -> ContieneCarne(?p)")

    Imp().set_as_rule("Pizza(?p), tiene_ingrediente(?p, ?i), IngredientePicante(?i) -> Picante(?p)")

    Imp().set_as_rule("ContieneCarne(?p), tiene_ingrediente(?p, ?i), Verdura(?i) -> PizzaMixta(?p)")

    Imp().set_as_rule("Pizza(?p), tiene_precio(?p, ?x), greaterThan(?x, 25.0) -> PizzaPremium(?p)")

In [10]:
# **COMPLETE** — crear los individuos:
#   Ingredientes:
#     - mozzarella, parmesano                      (Queso)
#     - jamon, carne_molida                        (Carne)
#     - pepperoni, chorizo                         (Carne; también IngredientePicante)
#     - tomate, albahaca, pina, maiz, cebolla      (Verdura)
#     - jalapeno                                   (Verdura; también IngredientePicante)
#     - salsa_tomate                               (Salsa)
#   Pizzas (con sus ingredientes y precio en miles de pesos colombianos):
#     - margarita   -> mozzarella, salsa_tomate, tomate, albahaca       precio = 18.0
#     - hawaiana    -> mozzarella, jamon, pina, salsa_tomate            precio = 22.0
#     - mexicana    -> mozzarella, carne_molida, jalapeno, tomate       precio = 26.0
#     - ranchera    -> mozzarella, chorizo, pepperoni, salsa_tomate     precio = 30.0
#     - andina      -> mozzarella, jalapeno, maiz, cebolla, salsa_tomate precio = 20.0


with piz:
    mozzarella   = Queso("mozzarella")
    parmesano    = Queso("parmesano")

    jamon        = Carne("jamon")
    carne_molida = Carne("carne_molida")

    pepperoni    = Carne("pepperoni")
    pepperoni.is_a.append(IngredientePicante)
    chorizo      = Carne("chorizo")
    chorizo.is_a.append(IngredientePicante)

    tomate       = Verdura("tomate")
    albahaca     = Verdura("albahaca")
    pina         = Verdura("pina")
    maiz         = Verdura("maiz")
    cebolla      = Verdura("cebolla")

    jalapeno     = Verdura("jalapeno")
    jalapeno.is_a.append(IngredientePicante)

    salsa_tomate = Salsa("salsa_tomate")

    margarita = Pizza("margarita")
    margarita.tiene_ingrediente = [mozzarella, salsa_tomate, tomate, albahaca]
    margarita.tiene_precio = 18.0

    hawaiana = Pizza("hawaiana")
    hawaiana.tiene_ingrediente = [mozzarella, jamon, pina, salsa_tomate]
    hawaiana.tiene_precio = 22.0

    mexicana = Pizza("mexicana")
    mexicana.tiene_ingrediente = [mozzarella, carne_molida, jalapeno, tomate]
    mexicana.tiene_precio = 26.0

    ranchera = Pizza("ranchera")
    ranchera.tiene_ingrediente = [mozzarella, chorizo, pepperoni, salsa_tomate]
    ranchera.tiene_precio = 30.0

    andina = Pizza("andina")
    andina.tiene_ingrediente = [mozzarella, jalapeno, maiz, cebolla, salsa_tomate]
    andina.tiene_precio = 20.0

## Razonar y verificar la clasificación

Llamen a `sync_reasoner_pellet` sobre la ontología de pizzas. Pellet disparará las cuatro reglas en cascada hasta que no haya nada nuevo que derivar.

> **Java necesario.** Pellet corre sobre Java; verifiquen `java -version` antes de invocar al razonador. Si falla con `OwlReadyJavaError`, ese es el síntoma.

Después de razonar, examinen para cada pizza qué clases tiene en `is_a`. La expectativa pedagógica es:

| Pizza | ContieneCarne | Picante | PizzaMixta | PizzaPremium |
|---|---|---|---|---|
| `margarita`   | no  | no  | no       | no       |
| `hawaiana`    | sí  | no  | **sí**   | no       |
| `mexicana`    | sí  | sí  | **sí**   | **sí**   |
| `ranchera`    | sí  | sí  | no       | **sí**   |
| `andina`      | no  | sí  | no       | no       |

Dos casos para mirar con cuidado:

- **`mexicana`** dispara las cuatro categorías. En particular `PizzaMixta` (R3) solo aparece porque antes apareció `ContieneCarne` (R1) — las reglas se alimentan unas a otras en cascada.
- **`ranchera`** lleva chorizo y pepperoni (ambos `Carne`), pero **no** lleva verdura: ningún ingrediente cumple el átomo `Verdura(?i)` de R3, así que aunque R1 dispare, R3 no encuentra cómo unificar y `ranchera` no queda como `PizzaMixta`.

### Instalar Java 25

In [11]:
!apt-get update -qq
!apt-get install -y openjdk-25-jre-headless
!ls /usr/lib/jvm

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Suggested packages:
  libnss-mdns fonts-dejavu-extra fonts-ipafont-gothic fonts-ipafont-mincho
  fonts-wqy-microhei | fonts-wqy-zenhei fonts-indic
The following NEW packages will be installed:
  openjdk-25-jre-headless
0 upgraded, 1 newly installed, 0 to remove and 64 not upgraded.
Need to get 59.0 MB of archives.
After this operation, 245 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 openjdk-25-jre-headless amd64 25.0.5.1+2-1~24.04.1 [59.0 MB]
Fetched 59.0 MB in 1s (77.0 MB/s)
Selecting previously unselected package openjdk-25-jre-headless:amd64.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack ...

In [12]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-25-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
!java -version

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "25.0.5.1" 2026-11-17
OpenJDK Runtime Environment (build 25.0.5.1+2-1-24.04.1-Ubuntu)
OpenJDK 64-Bit Server VM (build 25.0.5.1+2-1-24.04.1-Ubuntu, mixed mode, sharing)


In [13]:
# **COMPLETE** — invocar sync_reasoner_pellet sobre piz e imprimir,
#   por cada pizza, las cuatro categorías derivadas (ContieneCarne,
#   Picante, PizzaMixta, PizzaPremium) — sí o no.

with piz:
    sync_reasoner_pellet(infer_property_values=True, infer_data_property_values=True)

pizzas = [margarita, hawaiana, mexicana, ranchera, andina]
clases = [ContieneCarne, Picante, PizzaMixta, PizzaPremium]

print(f"{'Pizza':<10}" + "".join(f"{c.name:<15}" for c in clases))
for p in pizzas:
    fila = "".join(f"{('sí' if c in p.is_a else 'no'):<15}" for c in clases)
    print(f"{p.name:<10}{fila}")

* Owlready2 * Running Pellet...
    java -Xmx2000M -cp /usr/local/lib/python3.13/dist-packages/owlready2/pellet/log4j-api-2.19.0.jar:/usr/local/lib/python3.13/dist-packages/owlready2/pellet/antlr-3.2.jar:/usr/local/lib/python3.13/dist-packages/owlready2/pellet/xercesImpl-2.12.2.jar:/usr/local/lib/python3.13/dist-packages/owlready2/pellet/log4j-core-2.19.0.jar:/usr/local/lib/python3.13/dist-packages/owlready2/pellet/jena-tdb-0.10.0.jar:/usr/local/lib/python3.13/dist-packages/owlready2/pellet/log4j-1.2-api-2.19.0.jar:/usr/local/lib/python3.13/dist-packages/owlready2/pellet/aterm-java-1.6.jar:/usr/local/lib/python3.13/dist-packages/owlready2/pellet/pellet-2.3.1.jar:/usr/local/lib/python3.13/dist-packages/owlready2/pellet/slf4j-log4j12-1.6.4.jar:/usr/local/lib/python3.13/dist-packages/owlready2/pellet/owlapi-distribution-3.4.3-bin.jar:/usr/local/lib/python3.13/dist-packages/owlready2/pellet/httpclient-4.2.3.jar:/usr/local/lib/python3.13/dist-packages/owlready2/pellet/jena-iri-0.9.5.jar:/us

Pizza     ContieneCarne  Picante        PizzaMixta     PizzaPremium   
margarita no             no             no             no             
hawaiana  sí             no             sí             no             
mexicana  sí             sí             sí             sí             
ranchera  sí             sí             no             sí             
andina    no             sí             no             no             


* Owlready2 * Pellet took 1.8461072444915771 seconds
* Owlready * Reparenting pizzas.hawaiana: {pizzas.Pizza} => {pizzas.PizzaMixta, pizzas.ContieneCarne}
* Owlready * Reparenting pizzas.ranchera: {pizzas.Pizza} => {pizzas.PizzaPremium, pizzas.Picante, pizzas.ContieneCarne}
* Owlready * Reparenting pizzas.mexicana: {pizzas.Pizza} => {pizzas.PizzaPremium, pizzas.PizzaMixta, pizzas.Picante, pizzas.ContieneCarne}
* Owlready * Reparenting pizzas.andina: {pizzas.Pizza} => {pizzas.Picante}
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


## Preguntas analíticas

Respondan en celdas markdown nuevas:

1. **Decisión de modelado.** `tiene_ingrediente` es un `ObjectProperty`, no un `DataProperty`. ¿Por qué? ¿Qué pasaría si modelaran los ingredientes como cadenas de texto (`"queso"`, `"tomate"`)?
2. **Trazado del encadenamiento.** Para `mexicana`, ¿en qué orden lógico dispararon las reglas? ¿Por qué `PizzaMixta` *no podría* derivarse sin que antes haya disparado R1?
3. **El built-in `greaterThan`.** Cambien el umbral de R4 de `25` a `30` y vuelvan a razonar. ¿Quién deja de ser `PizzaPremium`? ¿Podrían lograr el mismo efecto sin built-ins, usando solo átomos de clase y propiedad?
4. **Límite expresivo.** Intenten escribir una regla `Imp` para `Vegetariana`, definida como "una pizza que **no** contiene carne". Las reglas `Imp` son **cláusulas de Horn positivas** (sin negación en el cuerpo): ¿pueden expresarla? Si no, ¿qué tipo de extensión lógica haría falta?

### Respuestas a las Preguntas Analíticas 1 y 2

**1. Decisión de modelado**
* **¿Por qué?** `tiene_ingrediente` conecta una instancia de `Pizza` con instancias de la clase `Ingrediente` (como `mozzarella`, `jamon`, etc.), los cuales son objetos/individuos dentro de la ontología y poseen su propia jerarquía de clases (`Queso`, `Carne`, `Verdura`). Al ser una relación entre dos individuos, debe ser un `ObjectProperty`.
* **¿Qué pasaría si se modelaran como cadenas de texto?** Si usáramos un `DataProperty` con valores como `"queso"` o `"tomate"`, los ingredientes serían simples literales (strings) sin semántica. El razonador no sabría que `"jamon"` es un tipo de `Carne` o que `"jalapeno"` es un `IngredientePicante`. Al perderse la jerarquía abstracta de los ingredientes, las reglas SWRL no podrían evaluar propiedades taxonómicas y la clasificación automática fallaría por completo.

**2. Trazado del encadenamiento para la pizza `mexicana`**
* **Orden lógico de disparo:**
  1. **R1** (`ContieneCarne`): Se dispara porque `mexicana` tiene el ingrediente `carne_molida`, que pertenece a la clase `Carne`.
  2. **R2** (`Picante`): Se dispara porque `mexicana` tiene el ingrediente `jalapeno`, que pertenece a `IngredientePicante`.
  3. **R4** (`PizzaPremium`): Se dispara porque el precio de `mexicana` ($26.0$) es mayor que $25.0$.
  4. **R3** (`PizzaMixta`): Se dispara al final, una vez que la regla R1 ya infirió y asignó la clase `ContieneCarne` a `mexicana`, cumpliendo con la precondición `ContieneCarne(?p)` de R3, junto con la presencia de `tomate` (que es `Verdura`).
* **¿Por qué `PizzaMixta` no podría derivarse sin R1?** R3 exige explícitamente el átomo `ContieneCarne(?p)` en su antecedente. Como nosotros no asignamos manualmente esa categoría a `mexicana` (solo le asignamos sus ingredientes concretos), la única manera de que la regla R3 unifique y se dispare es que R1 evalúe primero los ingredientes y clasifique a la pizza como `ContieneCarne`.

### Experimento pregunta 3: umbral de R4 en 30

In [14]:
for r in list(piz.rules()):
    if "greaterThan" in str(r):
        destroy_entity(r)

with piz:
    Imp().set_as_rule("Pizza(?p), tiene_precio(?p, ?x), greaterThan(?x, 30.0) -> PizzaPremium(?p)")

for p in pizzas:
    p.is_a = [Pizza]

with piz:
    sync_reasoner_pellet(infer_property_values=True, infer_data_property_values=True)

print(f"{'Pizza':<10}" + "".join(f"{c.name:<15}" for c in clases))
for p in pizzas:
    fila = "".join(f"{('sí' if c in p.is_a else 'no'):<15}" for c in clases)
    print(f"{p.name:<10}{fila}")

* Owlready2 * Running Pellet...
    java -Xmx2000M -cp /usr/local/lib/python3.13/dist-packages/owlready2/pellet/log4j-api-2.19.0.jar:/usr/local/lib/python3.13/dist-packages/owlready2/pellet/antlr-3.2.jar:/usr/local/lib/python3.13/dist-packages/owlready2/pellet/xercesImpl-2.12.2.jar:/usr/local/lib/python3.13/dist-packages/owlready2/pellet/log4j-core-2.19.0.jar:/usr/local/lib/python3.13/dist-packages/owlready2/pellet/jena-tdb-0.10.0.jar:/usr/local/lib/python3.13/dist-packages/owlready2/pellet/log4j-1.2-api-2.19.0.jar:/usr/local/lib/python3.13/dist-packages/owlready2/pellet/aterm-java-1.6.jar:/usr/local/lib/python3.13/dist-packages/owlready2/pellet/pellet-2.3.1.jar:/usr/local/lib/python3.13/dist-packages/owlready2/pellet/slf4j-log4j12-1.6.4.jar:/usr/local/lib/python3.13/dist-packages/owlready2/pellet/owlapi-distribution-3.4.3-bin.jar:/usr/local/lib/python3.13/dist-packages/owlready2/pellet/httpclient-4.2.3.jar:/usr/local/lib/python3.13/dist-packages/owlready2/pellet/jena-iri-0.9.5.jar:/us

Pizza     ContieneCarne  Picante        PizzaMixta     PizzaPremium   
margarita no             no             no             no             
hawaiana  sí             no             sí             no             
mexicana  sí             sí             sí             no             
ranchera  sí             sí             no             no             
andina    no             sí             no             no             


* Owlready2 * Pellet took 1.862368106842041 seconds
* Owlready * Reparenting pizzas.hawaiana: {pizzas.Pizza} => {pizzas.PizzaMixta, pizzas.ContieneCarne}
* Owlready * Reparenting pizzas.ranchera: {pizzas.Pizza} => {pizzas.Picante, pizzas.ContieneCarne}
* Owlready * Reparenting pizzas.mexicana: {pizzas.Pizza} => {pizzas.PizzaMixta, pizzas.Picante, pizzas.ContieneCarne}
* Owlready * Reparenting pizzas.andina: {pizzas.Pizza} => {pizzas.Picante}
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


3. El built-in greaterThan. Al subir el umbral de R4 de 25 a 30, mexicana (26.0) y ranchera (30.0) dejan de ser PizzaPremium, y ninguna pizza queda clasificada como premium. Ranchera sale porque greaterThan exige un valor estrictamente mayor, y 30.0 no es mayor que 30.0.

4. Límite expresivo. Una regla para Vegetariana sería: Pizza(?p), no ContieneCarne(?p) -> Vegetariana(?p). No se puede escribir con Imp, porque el cuerpo de una regla Imp es una conjunción de átomos positivos, es decir, una cláusula de Horn sin negación, y aquí haría falta negar ContieneCarne.

# III. RETROSPECTIVA

Responda las siguientes preguntas de forma individual y como equipo:

1. ¿Cuál fue el tiempo total invertido en el laboratorio por cada uno de ustedes? (Horas/Hombre)

Daniel Sua 4 horas

Juan Moreno 4 horas

2. ¿Cuál es el estado actual del laboratorio? ¿Por qué?

Consideramos que el laboratorio esta en estado completo, porque abordamos todo lo que se nos solicito para completarlo.

3. ¿Cuál consideran fue el mayor logro? ¿Por qué?

el mayor logro fue entender la nueva biblioteca y como esta se integra con con estos nuevos temas que no conociamos del todo

4. ¿Cuál consideran que fue el mayor problema técnico? ¿Qué hicieron para resolverlo?

el mayor problema tecnico fue revisar las versiones de java y python para que todo funcione correctamente.

5. ¿Qué hicieron bien como equipo? ¿Qué se comprometen a hacer para mejorar los resultados?

comunicacion para dividir y revisar lo realizado por el otro integrante.